# Exploratory Sensor-Physics Ablation: Gravity Removal on UCI-HAR (5 Folds)
Tests the channel-semantics hypothesis from Section IV-A / VI: does removing the gravity component from UCI-HAR (`total_acc` -> `0.3 Hz` high-pass `body_acc`, matching MotionSense `userAcceleration`) degrade stair-direction (`WALKING_UPSTAIRS` vs. `WALKING_DOWNSTAIRS`) separation under the $B=4$ Ranked-MNN promotion protocol?

**Note:** Explicitly labeled as an **exploratory post hoc sensor ablation**.
Switch Colab Runtime to **T4 GPU** if available, then click **Runtime -> Run all**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import butter, filtfilt
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist

PROJECT_DIR = Path("/content/drive/MyDrive/FedVocabHAR")
OUT_DIR = PROJECT_DIR / "artifacts" / "gravity_ablation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def find_file(filename):
    matches = list(PROJECT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"{filename} not found under {PROJECT_DIR}")
    return matches[0]

CACHE_PATH = find_file("ucihar_waist_totalacc_w128_s64_50hz.npz")
MANIFEST_PATH = find_file("phase5c_preregistered_manifest.json")
print("Cache:", CACHE_PATH)
print("Manifest:", MANIFEST_PATH)

In [ ]:
def remove_gravity_hpf(X, fs=50.0, cutoff=0.3, order=3):
    """Apply 0.3 Hz Butterworth high-pass filter to XYZ channels and recompute magnitude."""
    b, a = butter(order, cutoff / (0.5 * fs), btype="highpass")
    X_hpf = X.copy()
    # Filter each window's XYZ channels (shape N, 4, 128)
    for i in range(len(X_hpf)):
        for ch in range(3):
            X_hpf[i, ch] = filtfilt(b, a, X_hpf[i, ch], padlen=min(21, X_hpf.shape[-1] - 1))
    X_hpf[:, 3, :] = np.sqrt(np.sum(X_hpf[:, :3, :] ** 2, axis=1))
    return X_hpf.astype(np.float32)

class TemporalInceptionBlock(nn.Module):
    def __init__(self, in_ch, branch_ch=32, kernels=(9, 19, 39)):
        super().__init__()
        self.bottleneck = nn.Conv1d(in_ch, branch_ch, 1, bias=False)
        self.convs = nn.ModuleList([nn.Conv1d(branch_ch, branch_ch, k, padding=k // 2, bias=False) for k in kernels])
        self.pool = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(in_ch, branch_ch, 1, bias=False))
        out_ch = branch_ch * (len(kernels) + 1)
        self.bn = nn.BatchNorm1d(out_ch)
        self.res = nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch)) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        b = self.bottleneck(x)
        out = self.bn(torch.cat([c(b) for c in self.convs] + [self.pool(x)], dim=1))
        return F.relu(out + self.res(x))

class InceptionEncoder(nn.Module):
    def __init__(self, in_ch=4, num_known=4, emb_dim=64):
        super().__init__()
        self.b1 = TemporalInceptionBlock(in_ch)
        self.b2 = TemporalInceptionBlock(128)
        self.proj = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, emb_dim))
        self.head = nn.Linear(emb_dim, num_known)
    def encode(self, x):
        h = self.b2(self.b1(x)).mean(dim=-1)
        return F.normalize(self.proj(h), p=2, dim=-1)
    def forward(self, x):
        z = self.encode(x)
        return self.head(z), z

@torch.no_grad()
def encode_all(model, X, batch_size=256):
    model.eval()
    zs = []
    for i in range(0, len(X), batch_size):
        bx = torch.from_numpy(X[i:i+batch_size]).float().to(DEVICE)
        zs.append(model.encode(bx).cpu().numpy())
    return np.concatenate(zs, axis=0)

In [ ]:
def eval_f1(y_true, y_pred, classes):
    out = {}
    for c in classes:
        tp = int(np.sum((y_true == c) & (y_pred == c)))
        fp = int(np.sum((y_true != c) & (y_pred == c)))
        fn = int(np.sum((y_true == c) & (y_pred != c)))
        d = 2 * tp + fp + fn
        out[c] = (2.0 * tp / d) if d > 0 else 0.0
    return float(np.mean(list(out.values()))), out

def run_condition(label, X_all, y_str, subjects, manifest, epochs=20):
    known_names = manifest["known_activities"]
    novel_names = manifest["novel_activities"]
    all_names = known_names + novel_names
    name_to_id = {n: i for i, n in enumerate(all_names)}
    y = np.array([name_to_id[v] for v in y_str], dtype=np.int64)
    known_ids = [name_to_id[n] for n in known_names]
    novel_ids = [name_to_id[n] for n in novel_names]

    fold_rows = []
    for fold in manifest["outer_folds"]:
        torch.manual_seed(2026 + int(fold["fold"]))
        tr_s = set(map(int, fold["train_subjects"]))
        sel_s = set(map(int, fold["selection_subjects"]))
        int_s = sorted(set(map(int, fold["integration_subjects"])))
        te_s = set(map(int, fold["test_subjects"]))

        tr_idx = np.flatnonzero(np.isin(subjects, list(tr_s)) & np.isin(y, known_ids))
        model = InceptionEncoder(in_ch=4, num_known=len(known_ids), emb_dim=64).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=1e-4)
        ds = TensorDataset(torch.from_numpy(X_all[tr_idx]).float(), torch.from_numpy(y[tr_idx]).long())
        loader = DataLoader(ds, batch_size=64, shuffle=True)
        model.train()
        for _ in range(epochs):
            for bx, by in loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                opt.zero_grad()
                logits, _ = model(bx)
                F.cross_entropy(logits, by).backward()
                opt.step()

        Z = encode_all(model, X_all)
        # Known prototypes
        p_known = {}
        for c in known_ids:
            m = Z[tr_idx[y[tr_idx] == c]].mean(axis=0)
            p_known[c] = m / max(np.linalg.norm(m), 1e-12)
        P_k = np.stack([p_known[c] for c in known_ids], axis=0)

        # Calibration quantiles on train/selection
        d_sel = []
        for sid in sel_s:
            for c in known_ids:
                idx = np.flatnonzero((subjects == sid) & (y == c))[:80]
                if len(idx) >= 2:
                    d_sel.extend(pdist(Z[idx], metric="cosine").tolist())
        tau_loc = float(np.quantile(d_sel, 0.90))
        s_tr = []
        for c in known_ids:
            means = []
            for sid in tr_s:
                idx = np.flatnonzero((subjects == sid) & (y == c))
                if len(idx) > 0:
                    m = Z[idx].mean(axis=0)
                    means.append(m / max(np.linalg.norm(m), 1e-12))
            if len(means) >= 2:
                M_c = np.stack(means, axis=0)
                sims = M_c @ M_c.T
                s_tr.extend(sims[np.triu_indices(len(means), 1)].tolist())
        tau_agree = float(np.quantile(s_tr, 0.10))

        # Local novelty ranking + clustering on integration clients
        clusters = []
        for sid in int_s:
            idx = np.flatnonzero(subjects == sid)
            nov = 1.0 - np.max(Z[idx] @ P_k.T, axis=1)
            top = idx[np.argsort(-nov)[:min(600, len(idx))]]
            if len(top) < 12:
                continue
            dist_mat = pdist(Z[top], metric="cosine")
            Z_link = linkage(dist_mat, method="average")
            cids = fcluster(Z_link, t=tau_loc, criterion="distance")
            for cid in np.unique(cids):
                members = top[cids == cid]
                if len(members) >= 12:
                    sig = Z[members].mean(axis=0)
                    sig = sig / max(np.linalg.norm(sig), 1e-12)
                    medoid = int(members[np.argmax(Z[members] @ sig)])
                    clusters.append({"client": sid, "size": len(members), "sig": sig, "medoid": medoid})

        # MNN agreement
        best = {}
        for i, ca in enumerate(clusters):
            neigh = [(float(ca["sig"] @ cb["sig"]), j) for j, cb in enumerate(clusters) if cb["client"] != ca["client"] and float(ca["sig"] @ cb["sig"]) >= tau_agree]
            if neigh:
                best[i] = max(neigh)[1]
        pairs = []
        for i, j in best.items():
            if i < j and best.get(j) == i:
                g = clusters[i]["sig"] + clusters[j]["sig"]
                g = g / max(np.linalg.norm(g), 1e-12)
                v = i if float(clusters[i]["sig"] @ g) >= float(clusters[j]["sig"] @ g) else j
                pairs.append((clusters[i]["size"] + clusters[j]["size"], clusters[v]["medoid"], g))
        pairs.sort(key=lambda x: -x[0])

        def score_queries(q_list):
            proto = dict(p_known)
            for med, sig in q_list[:4]:
                lbl = int(y[med])
                if lbl in novel_ids and lbl not in proto:
                    proto[lbl] = sig
            avail = sorted(proto.keys())
            P_all = np.stack([proto[c] for c in avail], axis=0)
            te_idx = np.flatnonzero(np.isin(subjects, list(te_s)))
            preds = np.array([avail[i] for i in np.argmax(Z[te_idx] @ P_all.T, axis=1)], dtype=np.int64)
            nov_f1, nov_per = eval_f1(y[te_idx], preds, novel_ids)
            kn_f1, _ = eval_f1(y[te_idx], preds, known_ids)
            return nov_f1, kn_f1, {all_names[k]: v for k, v in nov_per.items()}

        mnn_nov, mnn_kn, mnn_per = score_queries([(m, g) for _, m, g in pairs])
        rng = np.random.default_rng(fold["fold"])
        rand_scores = []
        for _ in range(20):
            if clusters:
                pick = rng.choice(len(clusters), size=min(4, len(clusters)), replace=False)
                rn, _, _ = score_queries([(clusters[i]["medoid"], clusters[i]["sig"]) for i in pick])
            else:
                rn = 0.0
            rand_scores.append(rn)

        row = {
            "fold": fold["fold"],
            "mnn_novel_f1": mnn_nov,
            "mnn_known_f1": mnn_kn,
            "random_novel_f1": float(np.mean(rand_scores)),
            "mnn_beats_random": bool(mnn_nov > float(np.mean(rand_scores))),
            "per_novel_class": mnn_per,
        }
        fold_rows.append(row)
        print(f"{label} fold {fold['fold']}:", row)

    return {
        "condition": label,
        "mnn_novel_f1_mean": float(np.mean([r["mnn_novel_f1"] for r in fold_rows])),
        "random_novel_f1_mean": float(np.mean([r["random_novel_f1"] for r in fold_rows])),
        "mnn_known_f1_mean": float(np.mean([r["mnn_known_f1"] for r in fold_rows])),
        "fold_wins_vs_random": int(sum(r["mnn_beats_random"] for r in fold_rows)),
        "folds": fold_rows,
    }

data = np.load(CACHE_PATH, allow_pickle=True)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8-sig"))
X_total = data["x"].astype(np.float32)
if X_total.ndim == 3 and X_total.shape[-1] == 4 and X_total.shape[1] != 4:
    X_total = np.transpose(X_total, (0, 2, 1))
X_body = remove_gravity_hpf(X_total)

res_total = run_condition("uci_har_total_acc", X_total, data["activity"], data["subject"].astype(int), manifest)
res_body = run_condition("uci_har_body_acc_0.3hz_hpf", X_body, data["activity"], data["subject"].astype(int), manifest)
out = {"total_acc": res_total, "body_acc_hpf": res_body}
(OUT_DIR / "uci_har_gravity_ablation.json").write_text(json.dumps(out, indent=2))
print("SUMMARY:", json.dumps({k: {m: v[m] for m in ["mnn_novel_f1_mean", "random_novel_f1_mean", "mnn_known_f1_mean", "fold_wins_vs_random"]} for k, v in out.items()}, indent=2))